# Reproducing TimesNet (ICLR 2023) - Table 14: short-term forecasting (M4)


## 1. Wrap-UP Helper Functions

Includes functions whose purpose is to:

1) Set-Up Colab Environment Initialization and Avoid Collisions (e.x. Library Versions, GPUs, Folders etc.)

2) Wrap-Up Original Repository Code, Reproduce it and Formulate it Properly for Presentation and Evaluation (e.x. Seeds, Metrics, Discover Errors, Differences with the Paper Results etc.)

In [ ]:
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ---------------------------- Import Libraries -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

%%writefile repro_utils.py
import glob
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ----------------------------- Initializations -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

SEEDS = [2021, 2022, 2023]   # 2021 = TSLib's own seed; the paper averages 3 runs
SMOKE = False                # True = 1 training epoch per run, to test the pipeline
USE_DRIVE = False            # keep results on Google Drive (survives disconnects)
HF_REVISION = None           # None = latest on first run, then pinned automatically
TSLIB_URL = "https://github.com/thuml/Time-Series-Library.git"      # Source of Original code to clone
TSLIB_COMMIT = "4e938a1767106324dd753b2a44832bf870a0252e"           # Original Version of the code (2026-04-18)
HF_REPO = "thuml/Time-Series-Library"                               # Original Huggig Face dataset repository
DEFAULT_SEEDS = [2021, 2022, 2023]                                  # Original Seeds for Reprodusability

# Versions from TSLib's README / requirements.txt.
PINNED_PACKAGES = ["numpy==2.1.2",
                   "pandas==2.3.3",
                   "scipy==1.16.3",
                   "scikit-learn==1.7.2",
                   "matplotlib==3.10.8",
                   "sktime==0.40.1",
                   "patool==4.0.3",
                   "tqdm==4.66.5",
                   "datasets==4.5.0",
                   "huggingface_hub==0.36.0"]

TORCH_PROFILES = {"tslib": ("torch==2.5.1", "https://download.pytorch.org/whl/cu121", ["sympy==1.13.1"]),
                  "blackwell": ("torch==2.7.1", "https://download.pytorch.org/whl/cu128", [])}

_RUN_PY_SEED = "fix_seed = 2021"
_RUN_PY_SEED_NEW = "fix_seed = int(os.environ.get('TSLIB_SEED', '2021'))"
_RUN_PY_NP_SEED = "np.random.seed(fix_seed)"
_DETERMINISM_BLOCK = """np.random.seed(fix_seed)
    # [repro patch] deterministic kernels and no TF32 (TITAN RTX used in the paper has no TF32)
    if os.environ.get('TSLIB_DETERMINISTIC', '1') == '1':
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.use_deterministic_algorithms(True, warn_only=True)"""





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ---------------------------- Helper Functions -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Returns everything a shell command prints
def sh(cmd,
       cwd = None,
       env = None,
       check = True,
       quiet = False):

    # cmd: The command
    # cwd: Folder to run the command (cwd = None, runs it in the current folder of Colab)
    # env: Environment Variables
    # check: Check whether the command fails
    # quiet: Print the command, prefixed with "$"

    if env is None:
        env = dict(os.environ, MPLBACKEND="Agg")
    if not quiet:
        print("$", cmd)
    res = subprocess.run(cmd,
                         shell = True,
                         cwd = cwd,
                         env = env,
                         text = True,
                         stdout = subprocess.PIPE,
                         stderr = subprocess.STDOUT)
    if res.returncode != 0 and check:
        print(res.stdout[-4000:])
        raise RuntimeError(f"command failed ({res.returncode}): {cmd}")

    return res.stdout



# Returns True if the code can be run in Google Colab
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False



# Returns the folder, in which the reproduced table results are stored
def results_root(name,
                 use_drive = True):

    # name: The name of the table with respect to the ones in the paper (e.x. table 17)
    # use_drive: Whether to use Drive or Colab

    if in_colab() and use_drive:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        root = f"/content/drive/MyDrive/TimesNet_reproduction/{name}"
    else:
        base = "/content" if os.path.exists("/content") else os.getcwd()
        root = os.path.join(base, "TimesNet_reproduction", name)
    os.makedirs(root, exist_ok=True)

    return root



# Returns the content of a json file
def load_json(path,
              default = None):

    # path: Path of the file
    # default: Value to return, if the file doesn't exist

    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default



# Returns a JSON file
def save_json(path, obj):

    # path: Write the JSON file to "path"
    # obj: Which to write as a JSON file

    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ------------------------- Environment Functions ---------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Return the computational capability
def gpu_compute_capability():

    # e.x. 7.5 for T4, 8.0 for A100, 12.0 for RTX PRO 6000 (Colab G4)

    res = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
                         shell=True, text=True, capture_output=True)
    try:
        return float(res.stdout.strip().splitlines()[0]) if res.returncode == 0 else None
    except (ValueError, IndexError):
        return None



# Returns the path to the environment and it's description (as a dictionary)
def setup_environment(env_dir = "/content/tslib_env",
                      python_version = "3.11",
                      profile = None):

    # env_dir: Directory of the environment
    # python_version: Version of Python used
    # profile: The GPU

    cap = gpu_compute_capability()
    if profile is None:
        profile = "blackwell" if cap is not None and cap >= 10 else "tslib"
    if profile != "tslib":
        env_dir = f"{env_dir}_{profile}"
        print(f"NOTE: this GPU (compute capability {cap}) is newer than torch 2.5.1 supports, "
              f"so torch 2.7.1 (CUDA 12.8) is used instead of TSLib's recommended 2.5.1.\n"
              f"Runs stay repeatable on this GPU type; mention the version in your report, "
              f"or switch the runtime to T4/L4/A100/H100 to use TSLib's recommended version.")

    torch_pkg, index, extra = TORCH_PROFILES[profile]
    py = os.path.join(env_dir, "bin", "python")
    marker = os.path.join(env_dir, ".repro_ready")

    if not os.path.exists(marker):
        sh(f"{sys.executable} -m pip install -q uv")
        sh(f"uv venv --python {python_version} {env_dir}")
        sh(f"uv pip install --python {py} {torch_pkg} --index-url {index}")
        sh(f"uv pip install --python {py} " + " ".join(PINNED_PACKAGES + extra))
        open(marker, "w").close()

    info = sh(f"{py} -c \"import torch, numpy, pandas, sys; "
              f"print(sys.version.split()[0], torch.__version__, torch.version.cuda, "
              f"torch.cuda.is_available(), numpy.__version__, pandas.__version__)\"",
              quiet=True).split()

    env = {"python": info[0], "torch": info[1], "cuda": info[2],
           "cuda_available": info[3] == "True", "numpy": info[4], "pandas": info[5],
           "torch_profile": profile, "compute_capability": cap}
    print("Environment:", env)

    if cap is None or not env["cuda_available"]:
        print("WARNING: no GPU visible. In Colab: Runtime > Change runtime type > GPU.")
    else:
        test = subprocess.run(
            [py, "-c", "import torch; x = torch.randn(64, 64, device='cuda'); "
                       "print(float((x @ x).sum()), torch.cuda.get_device_name(0))"],
            text=True, capture_output=True)
        if test.returncode != 0:
            raise RuntimeError(
                "The GPU test failed, so training would fail too:\n"
                + test.stdout[-1500:] + test.stderr[-1500:]
                + "\nFix: Runtime > Change runtime type > T4, L4, A100 or H100, then run all cells.")
        print("GPU test passed:", test.stdout.strip().split(" ", 1)[-1])

    return py, env



# Returns the name of the GPU
def gpu_name():
    res = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                         text=True, capture_output=True)
    lines = res.stdout.strip().splitlines()
    return lines[0] if res.returncode == 0 and lines else "no GPU"





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# -------------------------- Repository Functions ---------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Ensures Reprodusability for the TSLib
def prepare_repo(repo_dir = "/content/Time-Series-Library",
                 commit = TSLIB_COMMIT):

    # repo_dir: Directory of the repository
    # commit: Version of the code to use

    if not os.path.exists(os.path.join(repo_dir, ".git")):
        sh(f"git clone -q {TSLIB_URL} {repo_dir}")
    sh(f"git -C {repo_dir} fetch -q origin {commit}", check=False, quiet=True)
    sh(f"git -C {repo_dir} checkout -q -f {commit}")
    head = sh(f"git -C {repo_dir} rev-parse HEAD", quiet=True).strip()
    assert head == commit, f"expected {commit}, got {head}"

    run_py = os.path.join(repo_dir, "run.py")
    with open(run_py) as f:
        src = f.read()
    if "[repro patch]" not in src:
        assert src.count(_RUN_PY_SEED) == 1 and src.count(_RUN_PY_NP_SEED) == 1, \
            "run.py layout changed; patches cannot be applied safely"
        src = src.replace(_RUN_PY_SEED, _RUN_PY_SEED_NEW)
        src = src.replace(_RUN_PY_NP_SEED, _DETERMINISM_BLOCK)
        with open(run_py, "w") as f:
            f.write(src)
    _patch_m4_numpy(repo_dir)
    print(f"TSLib ready at {repo_dir} (commit {commit[:10]}), run.py patched "
          f"(seed from TSLIB_SEED, deterministic kernels, TF32 off), M4 NumPy fix applied.")
    return repo_dir



#
_STACK_HELPER = '''
def _repro_stack(rows):
    """[repro patch] NumPy<1.24 behaviour of np.array(list_of_arrays): a 2-D array when all
    rows have the same length, otherwise a 1-D object array (NumPy>=1.24 raises instead)."""
    rows = list(rows)
    if len({len(r) for r in rows}) <= 1:
        return np.array(rows)
    out = np.empty(len(rows), dtype=object)
    out[:] = rows
    return out
'''



# Helps avoid TSLib's M4 code crash
def _patch_m4_numpy(repo_dir):
    """TSLib's M4 code builds arrays from series of different lengths. That raises an
    error on NumPy>=1.24 (TSLib pins 2.1.2), so M4 would crash on the real data.
    This restores the old NumPy behaviour exactly; nothing else changes."""
    targets = {
        "utils/m4_summary.py": [
            "return np.array([v[~np.isnan(v)] for v in values[groups == group_name]])",
            "naive2_forecasts = np.array([v[~np.isnan(v)] for v in naive2_forecasts])"],
        "data_provider/data_loader.py": [
            "training_values = np.array(\n            [v[~np.isnan(v)] for v in"],
    }
    for rel, olds in targets.items():
        path = os.path.join(repo_dir, rel)
        with open(path) as f:
            src = f.read()
        if "[repro patch] NumPy<1.24" in src:
            continue
        for old in olds:
            assert src.count(old) == 1, f"{rel}: unexpected layout, cannot patch"
            src = src.replace(old, old.replace("np.array(", "_repro_stack(", 1))
        anchor = "import numpy as np\n"
        assert anchor in src, f"{rel}: numpy import not found"
        src = src.replace(anchor, anchor + _STACK_HELPER, 1)
        with open(path, "w") as f:
            f.write(src)



# Returns the dataset used
def download_data(repo_dir,
                  python,
                  folders,
                  revision = None):

    # repo_dir: Directory of the repository
    # python: Version of Python
    # folders: Datasets folders to download
    # revision: Hugging Face revision

    """Download dataset folders from the TSLib Hugging Face repo into ./dataset.
    Returns the exact dataset revision (commit sha) used."""
    patterns = json.dumps([f"{d}/*" for d in folders])
    code = (
        "from huggingface_hub import snapshot_download, HfApi\n"
        f"rev = {revision!r} or HfApi().dataset_info({HF_REPO!r}).sha\n"
        f"snapshot_download(repo_id={HF_REPO!r}, repo_type='dataset', revision=rev,"
        f" allow_patterns={patterns}, local_dir='dataset')\n"
        "print('REVISION', rev)\n"
    )
    out = sh(f"{python} -c {shlex.quote(code)}", cwd=repo_dir, quiet=True)
    rev = re.search(r"REVISION (\S+)", out).group(1)
    missing = [d for d in folders if not os.listdir(os.path.join(repo_dir, "dataset", d))]
    if missing:
        raise RuntimeError(f"dataset folders empty after download: {missing}")
    print(f"Datasets {folders} ready (Hugging Face revision {rev[:10]}).")

    return rev



# Returns the current Colab session
def record_environment(root,
                       env,
                       commit,
                       data_rev):

    # root: Folder, in which the reproduced table results are stored
    # env: Environment Variables
    # commit: Version of the code to use
    # data_rev: Hugging Face Datasets to download

    """Store the environment; warn if this session's GPU differs from earlier runs."""
    path = os.path.join(root, "environment.json")
    info = load_json(path, {})
    current = {"gpu": gpu_name(), "tslib_commit": commit, "hf_dataset_revision": data_rev,
               **{f"env_{k}": v for k, v in env.items()}}
    runs_dir = os.path.join(root, "runs")
    has_runs = os.path.isdir(runs_dir) and any(f.endswith(".json") for f in os.listdir(runs_dir))
    if info and not has_runs:
        info = {}   # nothing finished yet: the new session simply becomes the reference
    if info:
        diffs = {k: (info.get(k), v) for k, v in current.items() if info.get(k) != v}
        if diffs:
            print("WARNING - this session differs from the one that produced earlier "
                  "results in this folder:")
            for k, (old, new) in diffs.items():
                print(f"   {k}: {old} -> {new}")
            print("Runs are only bit-for-bit repeatable on the same GPU model and "
                  "software. Use a new results folder or the same GPU type.")
    else:
        save_json(path, current)
    return current




# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# -------------------------------- Commands ---------------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Returns a list with one entry per training command
def expand_script(repo_dir,
                  script):

    # repo_dir: Directory of the Repository
    # script: Script to run

    shim = ("python() { for a in \"$@\"; do printf '%s\\037' \"$a\"; done; printf '\\036'; }; "
            "python3() { python \"$@\"; }; "
            f"source {shlex.quote(script)}")
    out = subprocess.run(["bash", "-c", shim], cwd=repo_dir, text=True,
                         capture_output=True, check=True).stdout
    commands = []
    for rec in out.split("\x1e"):
        parts = [p for p in rec.split("\x1f") if p != ""]
        if "run.py" in parts:
            commands.append(parts[parts.index("run.py") + 1:])

    return commands



# Returns the value that follows --<key> in an arg list
def arg(args,
        key,
        default = None):

    # args: The argument
    # key: The key
    # default: Returns None is the flag is absent

    flag = "--" + key
    return args[args.index(flag) + 1] if flag in args else default



# Returns a copy of the argument list
def set_arg(args,
            key,
            value):

    # args: Argument to return
    # key: The key to select
    # value: The value to set to "--<key>"

    args = list(args)
    flag = "--" + key
    if flag in args:
        args[args.index(flag) + 1] = str(value)
    else:
        args += [flag, str(value)]

    return args





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ----------------------------- Commands to Run -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Returns the TSLibs results
def run_tslib(python,
              repo_dir,
              args,
              seed,
              log_path,
              smoke = False):

    # python: Python version
    # repo_dir: Directory of the repository
    # args: Argument
    # seed: Seeds
    # log_path: Path logs
    # smoke: Whether to run a smoke test

    args = list(args)
    if smoke:
        args = set_arg(args, "train_epochs", 1)
    if os.environ.get("REPRO_FORCE_CPU") == "1":   # only for testing without a GPU
        args = args + ["--no_use_gpu", "--num_workers", "0"]
    env = dict(os.environ, TSLIB_SEED=str(seed), TSLIB_DETERMINISTIC="1",
               MPLBACKEND="Agg",   # Colab sets an inline backend the isolated env lacks
               CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONHASHSEED=str(seed),
               PYTHONUNBUFFERED="1")
    env.pop("CUDA_VISIBLE_DEVICES", None)
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    lines = []
    with open(log_path, "w") as log:
        log.write("python -u run.py " + " ".join(shlex.quote(a) for a in args) + "\n")
        log.write(f"# TSLIB_SEED={seed}\n")
        proc = subprocess.Popen([python, "-u", "run.py", *args], cwd=repo_dir, env=env,
                                text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            lines.append(line)
        proc.wait()

    return proc.returncode == 0, "".join(lines)



#
def clean_outputs(repo_dir,
                  keep_checkpoint_to = None):

    # repo_dir: Director of repository
    # keep_checkpoint_to: Checkpoints to select

    if keep_checkpoint_to:
        ckpts = sorted(glob.glob(os.path.join(repo_dir, "checkpoints", "*", "checkpoint.pth")),
                       key=os.path.getmtime)
        if ckpts:
            os.makedirs(os.path.dirname(keep_checkpoint_to), exist_ok=True)
            shutil.copy2(ckpts[-1], keep_checkpoint_to)
    for d in ["checkpoints", "results", "test_results"]:
        shutil.rmtree(os.path.join(repo_dir, d), ignore_errors=True)



# Prints logs of the execution time and seeds per dataset
def run_all(python,
            repo_dir,
            root,
            commands,
            seeds,
            parse,
            meta,
            smoke = False,
            key_fn = lambda a: arg(a, "model_id"),
            keep_checkpoints = False,
            after_run = None,
            max_commands = None):

    # python: Python version
    # repo_dir: Directory of the repository
    # root: Folder, in which the reproduced table results are stored
    # commands: Lists obtained from "expand_script"
    # seeds: Selected seeds
    # parse: Function that turns a run's output into a dictionary of metrics (e.x. accuracy)
    # meta: The record from "record_environment", copied into every result
    # smoke: Whether to run a smoke test or not
    # key_fn: Name of a run
    # keep_checkpoints: Whether to copy each of the model checkpoints into "checkpoints/" on the Drive or o Colab
    # after_run: Optional function to store predictions
    # max_commands: Number of commands to execute

    """Run every (command, seed) pair not already stored in `root/runs`.

    parse(output) -> dict of metrics or None; after_run(args, seed) is an optional hook.
    """
    runs_dir = os.path.join(root, "smoke_runs" if smoke else "runs")
    todo = commands[:max_commands] if max_commands else commands
    total, n = len(todo) * len(seeds), 0
    for seed in seeds:
        for args in todo:
            n += 1
            key = key_fn(args)
            out_json = os.path.join(runs_dir, f"{key}__seed{seed}.json")
            if os.path.exists(out_json):
                print(f"[{n}/{total}] {key} seed {seed}: already done, skipping")
                continue
            print(f"[{n}/{total}] {key} seed {seed}: running ...", end="", flush=True)
            t0 = time.time()
            ok, output = run_tslib(python, repo_dir, args, seed,
                                   os.path.join(root, "logs", f"{key}__seed{seed}.log"), smoke)
            metrics = parse(output) if ok else None
            if after_run:
                after_run(args, seed)
            ckpt = (os.path.join(root, "checkpoints", f"{key}__seed{seed}.pth")
                    if keep_checkpoints and ok and not smoke else None)
            clean_outputs(repo_dir, ckpt)
            minutes = (time.time() - t0) / 60
            if not metrics:
                print(f" FAILED after {minutes:.1f} min - see logs/{key}__seed{seed}.log")
                print(output[-2500:])
                continue
            save_json(out_json, {"key": key, "seed": seed, "metrics": metrics,
                                 "minutes": round(minutes, 2), "args": args,
                                 "smoke": smoke, **meta})
            shown = ", ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}"
                              for k, v in metrics.items())
            print(f" done in {minutes:.1f} min -> {shown}")



# Returns the reproduced table
def load_runs(root,
              smoke = False):

    # root: Root to select
    # smoke: Whether is smoke test or not

    import pandas as pd
    rows = []
    for p in glob.glob(os.path.join(root, "smoke_runs" if smoke else "runs", "*.json")):
        r = load_json(p)
        rows.append({"key": r["key"], "seed": r["seed"], **r["metrics"]})
    return pd.DataFrame(rows)



# Returns labels (matches, close, differs)
def verdict(delta,
            rel,
            kind):

    # delta: Difference in percentage
    # rel: Difference in percent of the paper's values
    # kind: Type of difference (e.x. absolute)

    if kind == "rel":
        a = abs(rel)
        return "✓ matches" if a <= 2 else ("~ close" if a <= 5 else "✗ differs")
    a = abs(delta)
    return "✓ matches" if a <= 1 else ("~ close" if a <= 3 else "✗ differs")



# Returns a check of determinism to define if the new result is identical to the stored one
def determinism_check(python,
                      repo_dir,
                      root,
                      args,
                      seed,
                      parse,
                      smoke = False):

    # python: Python version
    # repo_dir: Directory of the repository
    # root: Folder, in which the reproduced table results are stored
    # args: Arguments
    # seed: Seeds
    # parse: Function that turns a run's output into a dictionary of metrics (e.x. accuracy)
    # smoke: Whether it is a smoke test or not

    """Re-run one stored configuration and check the metric is identical."""
    key = arg(args, "model_id")
    stored = load_json(os.path.join(root, "smoke_runs" if smoke else "runs",
                                    f"{key}__seed{seed}.json"))
    if not stored:
        print("Run the experiments first."); return None
    ok, output = run_tslib(python, repo_dir, args, seed,
                           os.path.join(root, "logs", f"{key}__seed{seed}__recheck.log"), smoke)
    clean_outputs(repo_dir)
    again = parse(output) if ok else None
    same = again == stored["metrics"]
    print(f"stored : {stored['metrics']}\nre-run : {again}\n"
          + ("IDENTICAL - the run is bit-for-bit repeatable on this setup." if same else
             "DIFFERENT - check GPU model / environment against environment.json."))

    return same

Writing repro_utils.py


## 2. Environment, Original Code, Data, Optimizer Changes

Includes Notebook Cells whose purpose is to:

1) Create an Isolated Python 3.11 Environment (torch 2.5.1, CUDA 12.1 wheels, TSLib's pinned versions)

2) Clone TSLib at fixed commits

3) Downloads Original Datasets from Hugging Face


In [ ]:
import importlib, os, sys
sys.path.insert(0, os.getcwd())
import repro_utils as R
importlib.reload(R)
SEEDS, SMOKE, USE_DRIVE, HF_REVISION = R.SEEDS, R.SMOKE, R.USE_DRIVE, R.HF_REVISION

# Get ROOT
ROOT = R.results_root(name = 'table14',
                      use_drive = False)
print("Results folder:", ROOT)

# Get Python version and Environment
PY, ENV = R.setup_environment()          # Python 3.11 + torch 2.5.1 (cu121)

# Get TSLibRepository, with fixed commit
REPO = R.prepare_repo()
os.chdir(REPO)

# Get JSON
previous = R.load_json(os.path.join(ROOT, "environment.json"), {})

# Get Datasets
DATA_REV = R.download_data(REPO, PY, ['m4'],
                           revision=HF_REVISION or previous.get("hf_dataset_revision"))

# Get Colab Session
META = R.record_environment(root = ROOT,
                            env = ENV,
                            commit = R.TSLIB_COMMIT,
                            data_rev = DATA_REV)
print(META)

Results folder: /content/TimesNet_reproduction/table14
$ /usr/bin/python3 -m pip install -q uv
$ uv venv --python 3.11 /content/tslib_env
$ uv pip install --python /content/tslib_env/bin/python torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
$ uv pip install --python /content/tslib_env/bin/python numpy==2.1.2 pandas==2.3.3 scipy==1.16.3 scikit-learn==1.7.2 matplotlib==3.10.8 sktime==0.40.1 patool==4.0.3 tqdm==4.66.5 datasets==4.5.0 huggingface_hub==0.36.0 sympy==1.13.1
Environment: {'python': '3.11.17', 'torch': '2.5.1+cu121', 'cuda': '12.1', 'cuda_available': True, 'numpy': '2.1.2', 'pandas': '2.3.3', 'torch_profile': 'tslib', 'compute_capability': 8.9}
GPU test passed: NVIDIA L4
$ git clone -q https://github.com/thuml/Time-Series-Library.git /content/Time-Series-Library
$ git -C /content/Time-Series-Library checkout -q -f 4e938a1767106324dd753b2a44832bf870a0252e
TSLib ready at /content/Time-Series-Library (commit 4e938a1767), run.py patched (seed from TSLIB_SEED, dete

In [ ]:
p = os.path.join(REPO, "exp/exp_classification.py")
src = open(p).read()

# Switch from RAdam to Adam
old = "model_optim = optim.RAdam(self.model.parameters(), lr=self.args.learning_rate)"
new = "model_optim = optim.Adam(self.model.parameters(), lr=self.args.learning_rate)"

# Check if the changes have been applied
if old in src:
    open(p, "w").write(src.replace(old, new))
assert new in open(p).read(), "Adam patch not applied"
print("Optimizer: Adam (paper)")

Optimizer: Adam (paper)


## 3. Paper values (TimesNet column)

In [ ]:
# TimesNet column of Table 14: (SMAPE, MASE, OWA)
PAPER_T14 = {"Yearly": (13.387, 2.996, 0.786),
             "Quarterly": (10.100, 1.182, 0.890),
             "Monthly": (12.670, 0.933, 0.878),
             "Others": (4.891, 3.302, 1.035),
             "Average": (11.829, 1.585, 0.851)}

## 4. The Original Commands

Includes Notebook Cells whose purpose is to:

1) Use Commands from the Original Repository

2) Use Original Per-Dataset Initializations for the Model


In [ ]:
import re, json, shutil, pandas as pd

# Get Original Commands from repo
COMMANDS = R.expand_script(REPO, "scripts/short_term_forecast/TimesNet_M4.sh")
assert len(COMMANDS) == 6, len(COMMANDS)

# Check the M4 Naive2 file (needed for OWA) was downloaded
assert os.path.exists("dataset/m4/submission-Naive2.csv"), "M4 Naive2 file missing"

# Get official per-subset settings
pd.DataFrame([{k: R.arg(c, k) for k in ["model_id",
                                        "seasonal_patterns",
                                        "e_layers",
                                        "d_model",
                                        "d_ff",
                                        "top_k",
                                        "batch_size",
                                        "learning_rate",
                                        "train_epochs"]} for c in COMMANDS])

,model_id,seasonal_patterns,e_layers,d_model,d_ff,top_k,batch_size,learning_rate,train_epochs
0,m4_Monthly,Monthly,2,32,32,5,16,0.001,None
1,m4_Yearly,Yearly,2,16,32,5,16,0.001,None
2,m4_Quarterly,Quarterly,2,64,64,5,16,0.001,None
3,m4_Daily,Daily,2,16,16,5,16,0.001,None
4,m4_Weekly,Weekly,2,32,32,5,16,0.001,None
5,m4_Hourly,Hourly,2,32,32,5,16,0.001,None


In [ ]:
# Parameter d_ff is not in the paper.
PAPER_COMMANDS = []
for c in COMMANDS:
    for k, v in {"e_layers": 2,
                 "top_k": 5,
                 "d_model": 16,
                 "batch_size": 16,
                 "learning_rate": 0.001,
                 "train_epochs": 10}.items():
        c = R.set_arg(c, k, v)
    PAPER_COMMANDS.append(c)

# Get paper settings per subset
pd.DataFrame([{k: R.arg(c, k) for k in ["model_id",
                                        "seasonal_patterns",
                                        "e_layers",
                                        "d_model",
                                        "d_ff",
                                        "top_k",
                                        "batch_size",
                                        "learning_rate",
                                        "train_epochs"]} for c in PAPER_COMMANDS])

,model_id,seasonal_patterns,e_layers,d_model,d_ff,top_k,batch_size,learning_rate,train_epochs
0,m4_Monthly,Monthly,2,16,32,5,16,0.001,10
1,m4_Yearly,Yearly,2,16,32,5,16,0.001,10
2,m4_Quarterly,Quarterly,2,16,64,5,16,0.001,10
3,m4_Daily,Daily,2,16,16,5,16,0.001,10
4,m4_Weekly,Weekly,2,16,32,5,16,0.001,10
5,m4_Hourly,Hourly,2,16,32,5,16,0.001,10


## 5. Run

In [ ]:
# Folder for the forecasts of every seed
FORECASTS = os.path.join(ROOT, "smoke_forecasts" if SMOKE else "forecasts")

# A run succeeded if TSLib printed the test shape
def parse_m4(output):
    return {"status": "ok"} if "test shape" in output else None

# TSLib overwrites m4_results/<model>/ on every run, so move each seed's forecasts
def keep_forecast(args, seed):
    sp = R.arg(args, "seasonal_patterns").strip("'")
    src = os.path.join(REPO, "m4_results", "TimesNet", f"{sp}_forecast.csv")
    if os.path.exists(src):
        dst = os.path.join(FORECASTS, f"seed{seed}")
        os.makedirs(dst, exist_ok=True)
        shutil.move(src, os.path.join(dst, f"{sp}_forecast.csv"))

# Run Experiments
R.run_all(python = PY,
          repo_dir = REPO,
          root = ROOT,
          commands = PAPER_COMMANDS,
          seeds = SEEDS,
          parse = parse_m4,
          meta = META,
          smoke = SMOKE,
          after_run = keep_forecast)

# Compute SMAPE, MASE, OWA per seed
EVAL = '''
import json, sys
from utils.m4_summary import M4Summary
smape, owa, mape, mase = M4Summary(sys.argv[1], "./dataset/m4").evaluate()
print("M4JSON" + json.dumps({"smape": smape, "mase": mase, "owa": owa}, default=float))
'''
scores = {}
for seed in SEEDS:
    folder = os.path.join(FORECASTS, f"seed{seed}") + "/"
    if len(os.listdir(folder) if os.path.exists(folder) else []) < 6:
        print(f"seed {seed}: not all 6 subsets finished yet")
        continue
    out = R.sh(f"{PY} -c {R.shlex.quote(EVAL)} {folder}", cwd = REPO, quiet = True)
    scores[seed] = json.loads(out.split("M4JSON")[-1])
    R.save_json(os.path.join(ROOT, "smoke_scores" if SMOKE else "scores", f"seed{seed}.json"),
                scores[seed])
    print(f"seed {seed}: weighted average SMAPE {scores[seed]['smape']['Average']:.3f}, "
          f"MASE {scores[seed]['mase']['Average']:.3f}, OWA {scores[seed]['owa']['Average']:.3f}")

[1/18] m4_Monthly seed 2021: running ... done in 36.4 min -> status ok
[2/18] m4_Yearly seed 2021: running ... done in 12.2 min -> status ok
[3/18] m4_Quarterly seed 2021: running ... done in 18.9 min -> status ok
[4/18] m4_Daily seed 2021: running ... done in 3.3 min -> status ok
[5/18] m4_Weekly seed 2021: running ... done in 0.5 min -> status ok
[6/18] m4_Hourly seed 2021: running ... done in 0.5 min -> status ok
[7/18] m4_Monthly seed 2022: running ... done in 36.8 min -> status ok
[8/18] m4_Yearly seed 2022: running ... done in 12.2 min -> status ok
[9/18] m4_Quarterly seed 2022: running ... done in 18.8 min -> status ok
[10/18] m4_Daily seed 2022: running ... done in 3.3 min -> status ok
[11/18] m4_Weekly seed 2022: running ... done in 0.4 min -> status ok
[12/18] m4_Hourly seed 2022: running ... done in 0.6 min -> status ok
[13/18] m4_Monthly seed 2023: running ... done in 37.2 min -> status ok
[14/18] m4_Yearly seed 2023: running ... done in 13.5 min -> status ok
[15/18] m4_Qua

## 7. Comparison with the paper

In [9]:
assert scores, "No seed has all 6 subsets finished yet - run section 6 first."
rows = []
for group, (p_smape, p_mase, p_owa) in PAPER_T14.items():
    for metric, paper in [("SMAPE", p_smape), ("MASE", p_mase), ("OWA", p_owa)]:
        vals = {s: scores[s][metric.lower()][group] for s in scores}
        row = {"group": group, "metric": metric, "paper": paper,
               **{f"seed_{s}": v for s, v in vals.items()}}
        if vals:
            mean = sum(vals.values()) / len(vals)
            std = (sum((v - mean) ** 2 for v in vals.values()) / len(vals)) ** 0.5
            rel = 100 * (mean - paper) / paper
            row.update({"mean": mean, "std": std, "Δ": mean - paper, "Δ %": rel,
                        "verdict": R.verdict(None, rel, "rel")})
        rows.append(row)
table = pd.DataFrame(rows).set_index(["group", "metric"])
table.round(3).to_csv(os.path.join(ROOT, "table14_reproduction.csv"))
open(os.path.join(ROOT, "table14_reproduction.md"), "w").write(table.round(3).to_markdown())
table.round(3)

paper  seed_2021  seed_2022  seed_2023    mean    std  \
group     metric                                                           
Yearly    SMAPE   13.387     13.388     13.632     13.510  13.510  0.100   
          MASE     2.996      2.989      3.097      3.047   3.044  0.044   
          OWA      0.786      0.786      0.807      0.797   0.797  0.009   
Quarterly SMAPE   10.100     10.149     10.149     10.118  10.139  0.015   
          MASE     1.182      1.191      1.195      1.190   1.192  0.002   
          OWA      0.890      0.895      0.896      0.893   0.895  0.001   
Monthly   SMAPE   12.670     12.727     12.823     12.739  12.763  0.043   
          MASE     0.933      0.938      0.945      0.936   0.940  0.004   
          OWA      0.878      0.882      0.889      0.882   0.884  0.003   
Others    SMAPE    4.891      5.014      5.055      4.974   5.014  0.033   
          MASE     3.302      3.349      3.390      3.289   3.343  0.041   
          OWA      1.035      1.056      1.066      1.042   1.055  0.010   
Average   SMAPE   11.829     11.875     11.979     11.899  11.918  0.044   
          MASE     1.585      1.591      1.622      1.600   1.604  0.013   
          OWA      0.851      0.854      0.866      0.857   0.859  0.005   

                      Δ    Δ %    verdict  
group     metric                           
Yearly    SMAPE   0.123  0.919  ✓ matches  
          MASE    0.048  1.613  ✓ matches  
          OWA     0.011  1.357  ✓ matches  
Quarterly SMAPE   0.039  0.383  ✓ matches  
          MASE    0.010  0.846  ✓ matches  
          OWA     0.005  0.524  ✓ matches  
Monthly   SMAPE   0.093  0.734  ✓ matches  
          MASE    0.007  0.715  ✓ matches  
          OWA     0.006  0.721  ✓ matches  
Others    SMAPE   0.123  2.522    ~ close  
          MASE    0.041  1.232  ✓ matches  
          OWA     0.020  1.900  ✓ matches  
Average   SMAPE   0.089  0.750  ✓ matches  
          MASE    0.019  1.220  ✓ matches  
          OWA     0.008  0.940  ✓ matches

## 8. Determinism check
Re-runs one configuration and confirms the result is identical to the stored one.

In [ ]:
# Re-run the Yearly subset with seed 2021 and check if the forecasts are identical.
import filecmp
yearly = [c for c in COMMANDS if "Yearly" in R.arg(c, "seasonal_patterns")][0]
ok, _ = R.run_tslib(PY, REPO, yearly, SEEDS[0],
                    os.path.join(ROOT, "logs", "m4_Yearly__recheck.log"), SMOKE)
R.clean_outputs(REPO)
new = os.path.join(REPO, "m4_results", "TimesNet", "Yearly_forecast.csv")
old = os.path.join(FORECASTS, f"seed{SEEDS[0]}", "Yearly_forecast.csv")
same = ok and filecmp.cmp(new, old, shallow=False)
os.remove(new) if os.path.exists(new) else None
print("IDENTICAL forecasts - bit-for-bit repeatable." if same else
      "DIFFERENT - check GPU model / environment against environment.json.")

IDENTICAL forecasts - bit-for-bit repeatable.
